In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import random
from scipy import stats
from scipy.stats import fisher_exact
from scipy.stats.contingency import expected_freq

from localcider.sequenceParameters import SequenceParameters
import umap
MUNV940103 ={'A':1.08,'L':0.789,'R':0.976,'K':1.026,'N':1.197,'M':.812,'D':1.266,'F':0.685,
             'C':.733,'P':1.412,'Q':1.05,'S':0.987,'E':1.085,'T':0.784,'G':1.104,'W':0.755,'H':0.906,
             'Y':0.665,'I':0.583,'V':0.546}#from AAindex MUNV940103
hTMs = pd.read_pickle('../files/hTMs_uniprot_v1.pkl')
AF2 = pd.read_pickle('../files/AF2_with_ENSG.pkl')
mdSAMs = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')
mdSAMs = mdSAMs.drop_duplicates('AAseq')

In [ ]:
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr

In [ ]:
int(mdSAMs[mdSAMs.AAseq=='GSSVFLNRS'].start_new.values[0])

In [ ]:
AF2[AF2.acc=='Q8IX19'].AAseq.values[0]

In [ ]:
SSTMs = []
t= hTMs.acc.value_counts()
for i in t.index:
    if t[i]==1:
         SSTMs.append(i)
pd.Series(SSTMs).to_clipboard(index=0)

In [ ]:
SSTM_topology = pd.read_csv('/Users/aelazar/notebooks_for_paper/SSTM_data/idmapping_2023_06_28.tsv',delimiter='\t')

In [ ]:
SSTM_topology = pd.read_csv('/Users/aelazar/notebooks_for_paper/SSTM_data/idmapping_2023_06_28.tsv',delimiter='\t')
SSTM_topology = SSTM_topology[~pd.isna(SSTM_topology['Topological domain'])]
SSTM_topology = SSTM_topology[SSTM_topology['Topological domain'].str.find('Cytoplasmic')!=-1]

In [ ]:
SSTM_topology

In [ ]:
def mdSAM_loc_list(ind):

    acc = mdSAMs.loc[ind,'acc']
    print(acc)
    side = SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[1].split('"')[1]
    if SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('TOPO_DOM')[1].split('..')[0][1:].isnumeric()==True:
        s = int(SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('TOPO_DOM')[1].split('..')[0][1:])
        e = int(SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('TOPO_DOM')[1].split('..')[1].split(';')[0])
        #mdSAMs[mdSAMs.acc==acc].loc[4727,'start_new']                                     

        print('N term is %s'%side,s,e)                                                                                                        
        print('TM starts @ %s ends @ %s'%(hTMs[hTMs.acc==acc].start.values[0],hTMs[hTMs.acc==acc].end.values[0]))  


        mdSAM_s = mdSAMs.loc[ind,'start_new']
        print('mdSAM starts @ %d'%mdSAM_s)
        if mdSAM_s in range(s,e+1):
            dist= hTMs[hTMs.acc==acc].end.values[0]- mdSAM_s
            loc = SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[1].split('"')[1]
        else:
            dist = (mdSAM_s -hTMs[hTMs.acc==acc].end.values[0])
            loc = (SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[-1].split('"')[1])
        print(loc,dist)
        return loc,dist,side
    else:
        return np.NaN,np.NaN,np.NaN

In [ ]:
for i in mdSAMs[mdSAMs.acc.isin(SSTM_topology.From)].index:
    location, dist,side = mdSAM_loc_list(i)
    mdSAMs.loc[i,'location'] = location
    mdSAMs.loc[i,'dist'] = (dist)
    mdSAMs.loc[i,'N_side'] = side

In [ ]:
mdSAMs.to_pickle('mdSAM_loacation.pkl')

In [ ]:
mdSAMs

In [ ]:
SSTM_topology[SSTM_topology.From=='Q6DN72']['Transmembrane'].values[0].split('..')[0],

In [ ]:
x= pd.Series(SSTM_topology[SSTM_topology.From=='Q6DN72']['Topological domain'].values[0].split('TOPO_DOM '))
se = int(x[x.str.find('Extracellular')!=-1].values[0].split('..')[0])
ee = int(x[x.str.find('Extracellular')!=-1].values[0].split('..')[1].split(';')[0])
sc = int(x[x.str.find('Cytoplasmic')!=-1].values[0].split('..')[0])
ec = int(x[x.str.find('Cytoplasmic')!=-1].values[0].split('..')[1].split(';')[0])
print(sc,ec)

In [ ]:
AF2[AF2.acc=='Q6DN72']['pLDDT'].values[0][329:434]

In [ ]:
# pd.read_pickle('Balint_zscore_membrane_Neffecteve.pkl')

In [ ]:

def fisher_juxta_Cytoplasmic(acc,juxta_size):
    x= pd.Series(SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('TOPO_DOM '))
    #se = int(x[x.str.find('Extracellular')!=-1].values[0].split('..')[0])
    #ee = int(x[x.str.find('Extracellular')!=-1].values[0].split('..')[1].split(';')[0])
    sc = int(x[x.str.find('Cytoplasmic')!=-1].values[0].split('..')[0])
    ec = int(x[x.str.find('Cytoplasmic')!=-1].values[0].split('..')[1].split(';')[0])
    print(sc,ec)
    juxta_all = (len([i for i in pd.Series(AF2[AF2.acc==acc]['pLDDT'].values[0][sc:ec]).iloc[0:juxta_size] if i<70]))
    non_juxta_all  = (len([i for i in pd.Series(AF2[AF2.acc==acc]['pLDDT'].values[0][sc:ec]).iloc[juxta_size::] if i<70]))
    d_loc = []
    for i in mdSAMs[mdSAMs.acc==acc].index:
        for j in range(int(mdSAMs.loc[i,'start_new']),int(mdSAMs.loc[i,'end_new'])):
            d_loc.append(j)
    w = pd.Series(d_loc)
    juxa_mdSAM = len(w[w.between(sc,sc+juxta_size)])
    juxa_non_mdSAM = juxta_all-juxa_mdSAM
    non_juxta_mdSAM = len(w[w.between(sc+juxta_size+1,ec)])
    non_juxta_non_mdSAM = non_juxta_all-non_juxta_mdSAM
    #print(juxa_mdSAM,juxa_non_mdSAM,non_juxta_mdSAM,non_juxta_non_mdSAM)
    return juxa_mdSAM,juxa_non_mdSAM,non_juxta_mdSAM,non_juxta_non_mdSAM

In [ ]:
def fisher_juxta_Cytoplasmic_v2(acc,juxta_size):
    x= pd.Series(SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('TOPO_DOM '))
    #se = int(x[x.str.find('Extracellular')!=-1].values[0].split('..')[0])
    #ee = int(x[x.str.find('Extracellular')!=-1].values[0].split('..')[1].split(';')[0])
    sc = int(x[x.str.find('Cytoplasmic')!=-1].values[0].split('..')[0])
    ec = int(x[x.str.find('Cytoplasmic')!=-1].values[0].split('..')[1].split(';')[0])
    idr_regions= pd.Series([i[0] for i in enumerate(AF2[AF2.acc==acc]['pLDDT'].values[0]) if i[1]<70])
    idr_cyto = idr_regions[idr_regions.between(sc,ec)]
    juxa_s = sc+juxta_size
    juxa_e = sc+50+juxta_size
    #print(juxa_s,juxa_e)
    juxa_all = idr_cyto[idr_cyto.between(juxa_s,juxa_e)]
    idr_non_juxa = pd.Series(list(set.difference(set(idr_cyto),set(juxa_all))))
    #print(idr_non_juxa.sort_values())
    d_loc = []
    for i in mdSAMs[mdSAMs.acc==acc].index:
        for j in range(int(mdSAMs.loc[i,'start_new']),int(mdSAMs.loc[i,'end_new'])):
            d_loc.append(j)
    juxa_mdSAM = len(juxa_all[juxa_all.isin(d_loc)])
    juxa_non_mdSAM = len(juxa_all)-juxa_mdSAM
    non_juxa_mdSAM= len(idr_non_juxa[idr_non_juxa.isin(d_loc)])
    non_juxta_non_mdSAM = len(idr_non_juxa) - non_juxa_mdSAM
    return(juxa_mdSAM,juxa_non_mdSAM,non_juxa_mdSAM,non_juxta_non_mdSAM)

In [ ]:
fisher_juxta_Cytoplasmic_v2('O75197',10)

In [ ]:
pd.DataFrame(l)

In [ ]:

def fisher_juxta_Extracellular(acc,juxta_size):
    x= pd.Series(SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('TOPO_DOM '))
    se = int(x[x.str.find('Extracellular')!=-1].values[0].split('..')[0])
    ee = int(x[x.str.find('Extracellular')!=-1].values[0].split('..')[1].split(';')[0])
    len_ = ee-se
    
    juxta_all = (len([i for i in pd.Series(AF2[AF2.acc==acc]['pLDDT'].values[0][se:ee]).iloc[-juxta_size:] if i<70]))
    non_juxta_all  = (len([i for i in pd.Series(AF2[AF2.acc==acc]['pLDDT'].values[0][se:ee]).iloc[:(len_-juxta_size)] if i<70]))
    d_loc = []
    for i in mdSAMs[mdSAMs.acc==acc].index:
        for j in range(int(mdSAMs.loc[i,'start_new']),int(mdSAMs.loc[i,'end_new'])):
            d_loc.append(j)
    w = pd.Series(d_loc)
    juxa_mdSAM = len(w[w.between(ee-juxta_size,ee)])
    juxa_non_mdSAM = juxta_all-juxa_mdSAM
    non_juxta_mdSAM = len(w[w.between(se,ee-juxta_size)])
    non_juxta_non_mdSAM = non_juxta_all-non_juxta_mdSAM
    #print(juxa_mdSAM,juxa_non_mdSAM,non_juxta_mdSAM,non_juxta_non_mdSAM)
    return juxa_mdSAM,juxa_non_mdSAM,non_juxta_mdSAM,non_juxta_non_mdSAM

In [ ]:
idr_regions= [i[0] for i in enumerate(AF2[AF2.acc==acc]['pLDDT'].values[0]) if i[1]<70]
idr_cyto = pd.Series(idr_regions).between(se)

In [ ]:
from scipy.stats import fisher_exact,chi2_contingency
from scipy.stats.contingency import expected_freq
import tqdm

In [ ]:
slide = []
for juxa_size in tqdm.tqdm(range(1,200,25)):
    l = []
    for acc in set(mdSAMs[(mdSAMs.location=='Cytoplasmic')&(mdSAMs.N_side=='Extracellular')].acc):
        juxa_mdSAM,juxa_non_mdSAM,non_juxta_mdSAM,non_juxta_non_mdSAM = fisher_juxta_Cytoplasmic(acc=acc,juxta_size=juxa_size)
        l.append({'juxa_mdSAM':juxa_mdSAM,'juxa_non_mdSAM':juxa_non_mdSAM,
                  'non_juxta_mdSAM':non_juxta_mdSAM,'non_juxta_non_mdSAM':non_juxta_non_mdSAM,'acc':acc})
    table = pd.DataFrame(l).sum()
    #fisher_exact([[1341, 31584], [1295, 36946]], alternative='less')
    observed = np.array([[table.loc['juxa_mdSAM'], table.loc['juxa_non_mdSAM']], [table.loc['non_juxta_mdSAM'], table.loc['non_juxta_non_mdSAM']
    ]])
    res = chi2_contingency(observed)
    slide.append({'observed':observed[0][0],'exp':expected_freq(observed)[0][0],'obs_exp':np.log2(observed/expected_freq(observed))[0][0],'pval':res.pvalue,'juxa':juxa_size})

In [ ]:
slide = []
for juxa_size in tqdm.tqdm(range(0,200,10)):
    l = []
    for acc in set(mdSAMs[(mdSAMs.location=='Cytoplasmic')&(mdSAMs.N_side=='Extracellular')].acc):
        juxa_mdSAM,juxa_non_mdSAM,non_juxta_mdSAM,non_juxta_non_mdSAM = fisher_juxta_Cytoplasmic_v2(acc=acc,juxta_size=juxa_size)
        l.append({'juxa_mdSAM':juxa_mdSAM,'juxa_non_mdSAM':juxa_non_mdSAM,
                  'non_juxta_mdSAM':non_juxta_mdSAM,'non_juxta_non_mdSAM':non_juxta_non_mdSAM,'acc':acc})
    table = pd.DataFrame(l).sum()
    #fisher_exact([[1341, 31584], [1295, 36946]], alternative='less')
    observed = np.array([[table.loc['juxa_mdSAM'], table.loc['juxa_non_mdSAM']], [table.loc['non_juxta_mdSAM'], table.loc['non_juxta_non_mdSAM']
    ]])
    res = chi2_contingency(observed)
    slide.append({'observed':observed[0][0],'exp':expected_freq(observed)[0][0],'obs_exp':np.log2(observed/expected_freq(observed))[0][0],'pval':res.pvalue,'juxa':juxa_size})

In [ ]:
cyto_df = pd.DataFrame(slide)
cyto_df.sort_values(by='obs_exp')

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
sns.lineplot(y='observed',x='juxa',data=pd.DataFrame(slide),color='r')
sns.lineplot(y='exp',x='juxa',data=pd.DataFrame(slide))
#sns.lineplot(y='obs_exp',x='juxa',data=pd.DataFrame(slide))
#plt.yscale('log')
#plt.axvline(51)
plt.ylabel('mdSAM residues')
plt.xlabel('juxtamembrane width (IDR AAs)')

plt.axhline(0)
#plt.xscale('log')
#plt.yscale('log')

In [ ]:
pd.DataFrame(l).sum().iloc[:-1].sum()

In [ ]:
mdSAMs[mdSAMs.acc=='Q9ULI3']

In [ ]:
slide_Extracellular = []
for juxa_size in tqdm.tqdm(range(1,200,10)):
    l = []
    for acc in set(mdSAMs[(mdSAMs.location!='Cytoplasmic')&(mdSAMs.N_side=='Extracellular')].acc):
        juxa_mdSAM,juxa_non_mdSAM,non_juxta_mdSAM,non_juxta_non_mdSAM = fisher_juxta_Extracellular(acc=acc,juxta_size=juxa_size)
        l.append({'juxa_mdSAM':juxa_mdSAM,'juxa_non_mdSAM':juxa_non_mdSAM,
                  'non_juxta_mdSAM':non_juxta_mdSAM,'non_juxta_non_mdSAM':non_juxta_non_mdSAM,'acc':acc})
    table = pd.DataFrame(l).sum()
    #fisher_exact([[1341, 31584], [1295, 36946]], alternative='less')
    observed = np.array([[table.loc['juxa_mdSAM'], table.loc['juxa_non_mdSAM']], [table.loc['non_juxta_mdSAM'], table.loc['non_juxta_non_mdSAM']
    ]])
    res = chi2_contingency(observed)
    slide_Extracellular.append({'observed':observed[0][0],'exp':expected_freq(observed)[0][0],'obs_exp':np.log2(observed/expected_freq(observed))[0][0],'pval':res.pvalue,'juxa':juxa_size})

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
# sns.lineplot(y='observed',x='juxa',data=pd.DataFrame(slide_Extracellular),color='r')
# sns.lineplot(y='exp',x='juxa',data=pd.DataFrame(slide_Extracellular))
sns.lineplot(y='obs_exp',x='juxa',data=pd.DataFrame(slide_Extracellular))
#plt.yscale('log')
#plt.axvline(51)
plt.ylabel('mdSAM residues')
plt.xlabel('juxtamembrane width (IDR AAs)')

# plt.axhline(0)
#plt.xscale('log')
#plt.yscale('log')

In [ ]:
pd.DataFrame(l).sum().iloc[:-1].sum()

In [ ]:
mdSAMs.loc[:,['ENST','acc','AAseq','start_new','end_new','location','dist']]

In [ ]:
iPTM_phos = pd.read_pickle('iPTM_phos.pkl')

In [ ]:
import matplotlib

In [ ]:
cmap_phos = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#D0D3D6',"#18827E"])
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#18827E",'gray',"#B34F91"])#"#"

ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#BCBEC0","#3D8BD5"])

In [ ]:
phoso_aval = []
for AA in mdSAMs[mdSAMs.location=='Cytoplasmic'].AAseq:
    phoso_aval.append({'AAseq':AA,'Y':AA.count('Y'),'S':AA.count('S'),'T':AA.count('T')})
df_YST = pd.DataFrame(phoso_aval)
ax = sns.clustermap(df_YST.loc[:,['Y','S','T']],cmap=ww,figsize=(1.5,3),dendrogram_ratio=0.1,lw=0,method='average',col_cluster=False)
lab_x = [(str(i.get_text()).split()[0]) for i in ax.ax_heatmap.get_xticklabels()]
ax.ax_heatmap.set_yticklabels(['' for i in ax.ax_heatmap.get_yticklabels()],fontsize=0)
ax.ax_heatmap.set_xticklabels(['$%s$'%i for i in lab_x],fontsize=10)
ax.ax_cbar.set_ylabel("$No.$",size=10);
ax.ax_cbar.locator_params(nbins=5)
print(len(phoso_aval))
ax.ax_cbar.set_yticklabels(ax.ax_cbar.get_yticklabels(),fontsize=10)
plt.savefig('mdSAM_YST_counts_SSTM_cyto.svg',bbox_inches='tight',transparent=True)


In [ ]:
1- len(df_YST[(df_YST.Y==0)&(df_YST['T']==0)&(df_YST.S==0)])/len(df_YST)

In [ ]:
df_YST.Y.max()

In [ ]:
df_YST.iloc[:,1:].sum(axis=1).hist()

In [ ]:
phoso_aval = []
for AA in mdSAMs[mdSAMs.location=='Extracellular'].AAseq:
    phoso_aval.append({'AAseq':AA,'Y':AA.count('Y'),'S':AA.count('S'),'T':AA.count('T')})
df_YST = pd.DataFrame(phoso_aval)
ax = sns.clustermap(df_YST.loc[:,['S','T']],cmap=cmap_phos,figsize=(2,4),dendrogram_ratio=0.1,lw=0,method='average',col_cluster=False)
lab_x = [(str(i.get_text()).split()[0]) for i in ax.ax_heatmap.get_xticklabels()]
ax.ax_heatmap.set_yticklabels(['' for i in ax.ax_heatmap.get_yticklabels()],fontsize=0)
ax.ax_heatmap.set_xticklabels(['$%s$'%i for i in lab_x],fontsize=10)
ax.ax_cbar.set_ylabel("$No.$",size=10);
print(len(phoso_aval))
ax.ax_cbar.set_yticklabels(ax.ax_cbar.get_yticklabels(),fontsize=10)
plt.savefig('mdSAM_YST_counts_SSTM_non_cyto.svg',bbox_inches='tight',transparent=True)


In [ ]:
1- len(df_YST[(df_YST['T']==0)&(df_YST.S==0)])/len(df_YST)

In [ ]:
t = hTMs['acc'].value_counts()
for i in mdSAMs.index:
    mdSAMs.loc[i,'no_TMs']= t[mdSAMs.loc[i,'acc']]
mdSAMs[(mdSAMs.dist>0)&(mdSAMs.location.isin(['Cytoplasmic','Extracellular','Lumenal']))].location.dropna().value_counts()
l = []
for i in range(437):
    l.append({'val':1,'type':'Cytoplasmic'})
for i in range(136):
    l.append({'val':1,'type':'Extracellular'})
for i in range(49):
    l.append({'val':1,'type':'Lumenal'})

In [ ]:
# fig, ax = plt.subplots(figsize=(3,4)) 
f, (ax, ax2) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [5,1]},figsize=(3, 4))
sns.boxenplot(x='location',y='dist',data=mdSAMs[(mdSAMs.location.isin(['Cytoplasmic','Extracellular','Lumenal']))&(mdSAMs.dist>0)],log_scale=True,hue='location',
             palette=['#232323','#043278','#397BDE'],hue_order=['Cytoplasmic','Extracellular','Lumenal'],width=0.75,ax=ax)
ax.set_ylabel('No. amino acids (to closest leaflet)',fontsize=14)
ax.set_xticklabels(['CP','EC','Lumenal'],fontsize=14)
#plt.xlabel('')
ax.grid()
ax.set_xlabel('mdSAM location',fontsize=16)
sns.histplot(x='val',data=pd.DataFrame(l),ax=ax2
                  ,hue='type',multiple="stack",lw=.5,bins=1,alpha=1,
                  edgecolor='w',zorder=10,palette=['#232323','#043278','#397BDE'])
ax2.set_xlabel('')
ax2.set_ylabel('Count',fontsize=12)
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0,fontsize=12)
ax2.set_xticklabels([])
ax2.locator_params(axis='y', nbins=6)
ax2.legend([],frameon=False)
plt.savefig('mdSAM_dist_v4.svg',bbox_inches='tight',transparent=True)


In [ ]:
f, (ax, ax2) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [5,1]},figsize=(3, 4))
sns.violinplot(y='dist',fill=False,data=mdSAMs[(mdSAMs.location.isin(['Cytoplasmic','Extracellular','Lumenal']))&(mdSAMs.dist>0)],log_scale=True,
              split=False,inner='box',width=0.8,x='location',palette=['#8C8C8C','#043278','#397BDE'],ax=ax)#,inner_kws=dict(box_width=10))#,order=['Cytoplasmic','Extracellular','Lumenal'])
ax.set_ylabel('No. amino acids (to closest leaflet)',fontsize=14)
ax.set_xticklabels(['CP','EC','Lumenal'],fontsize=14)
ax.set_ylim([1,2e4])

#plt.xlabel('')

ax.yaxis.set_tick_params(labelsize = 12)
ax.set_xlabel('mdSAM location',fontsize=16)
sns.histplot(x='val',data=pd.DataFrame(l),ax=ax2
                  ,hue='type',multiple="stack",lw=.5,bins=1,alpha=1,
                  edgecolor='w',zorder=10,palette=['#8C8C8C','#043278','#397BDE'])
ax2.set_xlabel('')
ax.set_xticklabels([])
ax.set_xlabel('')
ax.set_ylabel('')
ax2.set_ylabel('Count',fontsize=14)
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0,fontsize=12)
ax2.set_xticklabels([])
ax2.locator_params(axis='y', nbins=6)
ax2.legend([],frameon=False)
plt.savefig('mdSAM_dist_v5.svg',bbox_inches='tight',transparent=True)


In [ ]:
len(pd.DataFrame(l)[pd.DataFrame(l)['type']=='Extracellular'])

In [ ]:
sns.kdeplot(x='dist',data=mdSAMs[mdSAMs.location.isin(['Cytoplasmic','Extracellular','Lumenal'])],
            log_scale=True,hue='location',common_norm=False,linewidths=0,
            fill=True,alpha=.5,palette=['#232323','#043278','#397BDE'],hue_order=['Cytoplasmic','Extracellular','Lumenal'],)

plt.xlim([1,2e4])

In [ ]:
sns.kdeplot(mdSAMs[mdSAMs.location.isin(['Cytoplasmic'])].dist,log_scale=True)
sns.kdeplot(mdSAMs[mdSAMs.location.isin(['Extracellular'])].dist,log_scale=True)
sns.kdeplot(mdSAMs[mdSAMs.location.isin(['Lumenal'])].dist,log_scale=True)

In [ ]:
fig = plt.figure(figsize = ((3.75),5))#20,10
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
axs = fig.subplots(nrows=3, ncols=1,gridspec_kw = {'hspace':0,'wspace':-.1})

sns.kdeplot(mdSAMs[mdSAMs.location.isin(['Cytoplasmic'])].dist,log_scale=True,ax=axs[0],fill=True,color='#B96071',lw=0,alpha=.9)#043278
sns.kdeplot(mdSAMs[mdSAMs.location.isin(['Extracellular'])].dist,log_scale=True,ax=axs[1],fill=True,color='#679AAB',lw=0,alpha=.9)#232323
sns.kdeplot(mdSAMs[mdSAMs.location.isin(['Lumenal'])].dist,log_scale=True,ax=axs[2],fill=True,color='#624A64',lw=0,alpha=.9)#043278
for i in [0,1,2]:
    axs[i].set_ylim([0,.8])
    axs[i].set_xlim([1,10000])
axs[0].set_xticklabels([])
axs[1].set_xticklabels([])
axs[0].set_yticklabels([])
axs[1].set_yticklabels([])
axs[1].set_ylabel('')
axs[0].set_ylabel('')
axs[1].set_xlabel('')
axs[0].set_xlabel('')
plt.savefig('mdSAM_dist_v2.svg',bbox_inches='tight',transparent=True)
for i in [0,1,2]:
    axs[i].spines[['right', 'top','right','left']].set_visible(False)
    axs[i].set_yticklabels([])
    axs[i].set_xlabel('')
    axs[i].set_ylabel('')
axs[2].set_xlabel('No. amino acids\n(to closest leaflet)',fontsize=14)
plt.savefig('mdSAM_dist_v2.svg',bbox_inches='tight',transparent=True)


In [ ]:
fig, ax = plt.subplots(figsize=(.5,3)) 
plt.locator_params(axis='y', nbins=6)
sns.histplot(x='val',data=pd.DataFrame(l)
                  ,hue='type',multiple="stack",lw=.5,bins=1,alpha=1,
                  edgecolor='w',zorder=10,palette=['#B96071','#679AAB','#624A64'])
ax.set_xlabel('')
ax.set_ylabel('Count',fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels([])
plt.legend([],frameon=False)
plt.savefig('mdSAM_dist_v3.svg',bbox_inches='tight',transparent=True)


In [ ]:
fig = plt.figure(figsize = ((3.75),5))#20,10
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
axs = fig.subplots(nrows=3, ncols=1,gridspec_kw = {'hspace':0,'wspace':-.1})

sns.kdeplot(mdSAMs[mdSAMs.location.isin(['Cytoplasmic'])].dist,log_scale=True,ax=axs[0],fill=True,color='#232323',lw=0,alpha=.9)#
sns.kdeplot(mdSAMs[mdSAMs.location.isin(['Extracellular'])].dist,log_scale=True,ax=axs[1],fill=True,color='#043278',lw=0,alpha=.9)#
sns.kdeplot(mdSAMs[mdSAMs.location.isin(['Lumenal'])].dist,log_scale=True,ax=axs[2],fill=True,color='#397BDE',lw=0,alpha=.9)#
for i in [0,1,2]:
    axs[i].set_ylim([0,.8])
    axs[i].set_xlim([1,10000])
axs[0].set_xticklabels([])
axs[1].set_xticklabels([])
axs[0].set_yticklabels([])
axs[1].set_yticklabels([])
axs[1].set_ylabel('')
axs[0].set_ylabel('')
axs[1].set_xlabel('')
axs[0].set_xlabel('')
plt.savefig('mdSAM_dist_v2.svg',bbox_inches='tight',transparent=True)
for i in [0,1,2]:
    axs[i].spines[['right', 'top','right','left']].set_visible(False)
    axs[i].set_yticklabels([])
    axs[i].set_xlabel('')
    axs[i].set_ylabel('')
axs[2].set_xlabel('No. amino acids\n(to closest leaflet)',fontsize=14)
plt.savefig('mdSAM_dist_v2.svg',bbox_inches='tight',transparent=True)


In [ ]:
fig, ax = plt.subplots(figsize=(.5,3)) 
plt.locator_params(axis='y', nbins=6)
sns.histplot(x='val',data=pd.DataFrame(l)
                  ,hue='type',multiple="stack",lw=.5,bins=1,alpha=1,
                  edgecolor='w',zorder=10,palette=['#B96071','#679AAB','#624A64'])
ax.set_xlabel('')
ax.set_ylabel('Count',fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels([])
plt.legend([],frameon=False)
plt.savefig('mdSAM_dist_v3.svg',bbox_inches='tight',transparent=True)


In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 

ax = sns.histplot(x='dist',data=mdSAMs[mdSAMs.location.isin(['Cytoplasmic','Extracellular','Lumenal'])]
                  ,hue='location',multiple="stack",log_scale=(True),lw=.4,bins=15,alpha=1,
                  edgecolor='w',zorder=10,palette=['#B34F91','#18827E','#011627'])#,palette=['#E67D79','#8CC6EC','#C0C0BE']
legend = ax.get_legend()
handles = legend.legendHandles
legend.remove()
ax.legend(handles, ['$Cytoplasmic$', '$Extracellular$', '$Lumenal$'], title='',fontsize=8)
left, bottom, width, height = [0.8, 0.25, 0.06, 0.65]
ax2 = fig.add_axes([left, bottom, width, height])
sns.histplot(x='val',data=pd.DataFrame(l)
                  ,hue='type',multiple="stack",lw=.5,bins=1,alpha=1,
                  edgecolor='w',zorder=10,palette=['#B34F91','#18827E','#011627'],ax=ax2)
ax2.spines[['right', 'top']].set_visible(True)
ax2.set_xticks([])
ax2.legend(frameon=False)
#ax2.set_yticks([])
ax2.set_xlabel('')
ax2.set_ylabel('')
ax2.tick_params(axis='both', which='major', labelsize=8)
ax2.yaxis.set_major_locator(plt.MaxNLocator(3))
ax.grid()
ax.set_xlabel('$No.\;amino\;acids$\n$(to\;closest\;leaflet)$',fontsize=10)
ax.set_xlabel('No. amino acids\n(to closest leaflet)',fontsize=10)

ax.set_ylabel('Count',fontsize=10)
plt.savefig('mdSAM_dist.svg',bbox_inches='tight',transparent=True)


In [ ]:
pd.DataFrame(l)['type'].value_counts()

In [ ]:
mdSAMs[mdSAMs.location.isin(['Cytoplasmic','Extracellular','Lumenal'])].dist.describe()

In [ ]:
uniport_annotation  = pd.read_csv('/Users/aelazar/Downloads/idmapping_2023_11_15.tsv',delimiter='\t')
OD = set(uniport_annotation[uniport_annotation['Gene Ontology IDs'].str.find('GO:0004984')!=-1].Entry)# PM GO 
t = hTMs['acc'].value_counts().reset_index()
mdSAMs_no_OD = mdSAMs[~mdSAMs.acc.isin(OD)]
t_no_OD = t[~t['index'].isin(OD)]
t_no_OD.head(1)

In [ ]:
d = []
v = len(mdSAMs_no_OD.drop_duplicates(subset='acc')[mdSAMs_no_OD.no_TMs.between(1,1)])
d.append({'n':'1','val':v,'norm_v':v/len(mdSAMs_no_OD.drop_duplicates(subset='acc')),'ele':'mdSAM'})

v = len(mdSAMs_no_OD.drop_duplicates(subset='acc')[mdSAMs_no_OD.no_TMs.between(2,6)])
d.append({'n':'2-6','val':v,'norm_v':v/len(mdSAMs_no_OD.drop_duplicates(subset='acc')),'ele':'mdSAM'})

v = len(mdSAMs_no_OD.drop_duplicates(subset='acc')[mdSAMs_no_OD.no_TMs==7])
d.append({'n':'7','val':v,'norm_v':v/len(mdSAMs_no_OD.drop_duplicates(subset='acc')),'ele':'mdSAM'})

v = len(mdSAMs_no_OD.drop_duplicates(subset='acc')[mdSAMs_no_OD.no_TMs>7])
d.append({'n':'>7','val':v,'norm_v':v/len(mdSAMs_no_OD.drop_duplicates(subset='acc')),'ele':'mdSAM'})

v = len(t_no_OD[(~t_no_OD['index'].isin(mdSAMs_no_OD.acc))&(t_no_OD['acc'].between(1,1))])
d.append({'n':'1','val':v,'norm_v':v/len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)]),'ele':'non_mdSAM'})

v = len(t_no_OD[(~t_no_OD['index'].isin(mdSAMs_no_OD.acc))&(t_no_OD['acc'].between(2,6))])
d.append({'n':'2-6','val':v,'norm_v':v/len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)]),'ele':'non_mdSAM'})

v = len(t_no_OD[(~t_no_OD['index'].isin(mdSAMs_no_OD.acc))&(t_no_OD['acc'].between(7,7))])
d.append({'n':'7','val':v,'norm_v':v/len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)]),'ele':'non_mdSAM'})

v = len(t_no_OD[(~t_no_OD['index'].isin(mdSAMs_no_OD.acc))&(t_no_OD['acc'].between(7,100))])
d.append({'n':'>7','val':v,'norm_v':v/len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)]),'ele':'non_mdSAM'})
number_of_TMs = pd.DataFrame(d)
l_fisher = []
for i in set(number_of_TMs.n):
    a = number_of_TMs[(number_of_TMs.n==i)&(number_of_TMs.ele=='mdSAM')].val.values[0]
    b = number_of_TMs[(number_of_TMs.ele=='mdSAM')].val.sum()-a
    c = number_of_TMs[(number_of_TMs.n==i)&(number_of_TMs.ele=='non_mdSAM')].val.values[0]
    e = number_of_TMs[(number_of_TMs.ele=='non_mdSAM')].val.sum()-c
    table = [[a,b],[c,e]]
    res = fisher_exact(table)
    expected_freq(table)
    # print(i)
    # print(table[0][0],expected_freq(table)[0][0])
    # print(res)
    l_fisher.append({'n':i,'val':table[0][0],'ele':'obs','pval':res[-1]})
    l_fisher.append({'n':i,'val':expected_freq(table)[0][0],'ele':'exp','pval':res[-1]})
df_fisher = pd.DataFrame(l_fisher)
print(df_fisher)

In [ ]:
t = hTMs['acc'].value_counts()
d = []
for i in range(1,8):
    v = len(mdSAMs_no_OD.drop_duplicates(subset='acc')[mdSAMs_no_OD.no_TMs==i])
    d.append({'n':'%d'%i,'val':v,'norm_v':v/len(mdSAMs_no_OD.drop_duplicates(subset='acc')),'ele':'mdSAM'})
v = len(mdSAMs_no_OD.drop_duplicates(subset='acc')[mdSAMs_no_OD.no_TMs==i])
d.append({'n':'>7','val':v,'norm_v':v/len(mdSAMs_no_OD.drop_duplicates(subset='acc')),'ele':'mdSAM'})
for i in range(1,8):
    v = len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)][t==i])
    d.append({'n':'%d'%i,'val':v,'norm_v':v/len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)]),'ele':'non_mdSAM'})
v = len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)][t>7])
d.append({'n':'>7','val':v,'norm_v':v/len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)]),'ele':'non_mdSAM'})
number_of_TMs = pd.DataFrame(d)
l_fisher = []
for i in set(number_of_TMs.n):
    a = number_of_TMs[(number_of_TMs.n==i)&(number_of_TMs.ele=='mdSAM')].val.values[0]
    b = number_of_TMs[(number_of_TMs.ele=='mdSAM')].val.sum()-a
    c = number_of_TMs[(number_of_TMs.n==i)&(number_of_TMs.ele=='non_mdSAM')].val.values[0]
    e = number_of_TMs[(number_of_TMs.ele=='non_mdSAM')].val.sum()-c
    table = [[a,b],[c,e]]
    res = fisher_exact(table)
    expected_freq(table)
    # print(i)
    # print(table[0][0],expected_freq(table)[0][0])
    # print(res)
    l_fisher.append({'n':i,'val':table[0][0],'ele':'obs','pval':res[-1]})
    l_fisher.append({'n':i,'val':expected_freq(table)[0][0],'ele':'exp','pval':res[-1]})
df_fisher = pd.DataFrame(l_fisher)
print()

In [ ]:
number_of_TMs

In [ ]:
fig, ax = plt.subplots(figsize=(3.3,3.3)) 
sns.barplot(x='n',y='val',data=number_of_TMs,hue='ele',zorder=10)
# ax.grid()
# df_fisher = pd.DataFrame(l_fisher)
# df_fisher['fdr'] = fdr(df_fisher['pval'])
# df_fisher[df_fisher.fdr>.05]
# fig, ax = plt.subplots(figsize=(2.5*1.618,2.5)) 
# sns.barplot(x='n',y='val',data=df_fisher,hue='ele',zorder=10,order=['1', '2', '3', '4', '5', '6', '7', '>7']
#            ,hue_order=['obs','exp'],lw=0,palette=['#2626A7','#939598'])
# plt.ylim([0,600])
# ax.text(-.25,550,'***')
# ax.text(5.75,200,'***')
# ax.text(7.0,150,'*')
# plt.xlabel('No. TM segments',fontsize=10)
# plt.ylabel('No. with mdSAM',fontsize=10)
# ax.grid()
#plt.savefig('TM_segments.svg',bbox_inches='tight',transparent=True)



In [ ]:
print(number_of_TMs[number_of_TMs.n.isin(['2','3','4','5','6'])].val.sum())
print(number_of_TMs[number_of_TMs.n.isin(['7'])].val.sum())
print(number_of_TMs[number_of_TMs.n.isin(['>7'])].val.sum())
print(number_of_TMs[number_of_TMs.n.isin(['1'])].val.sum())
t = t.reset_index() 
t_no_OD = t[~t['index'].isin(OD)]
t_no_OD.head()

In [ ]:
df_fisher

In [ ]:
fig, ax = plt.subplots(figsize=(4,4)) 
plt.locator_params(axis='y', nbins=5)
bar = sns.barplot(x='n',y='val',data=df_fisher,hue='ele',zorder=10,hue_order=['obs','exp'],lw=0,palette=['#2626A7','#939598'],
                 order=['1','2-6','7','>7'],width=.95)
hatches = ['\\\\' for i in range(12)]

# # Loop over the bars
c = 0
colors = ['#666666','#DA726C','#CE433B','#832621','#666666','#DA726C','#CE433B','#832621']
for i,thisbar in enumerate(bar.patches):
    # Set a different hatch for each bar
    if i in [4,5,6,7]:
        
        thisbar.set_facecolor(colors[c])
        thisbar.set_hatch(hatches[i])
        thisbar.set_edgecolor('white')
        c = c+1
c = 0
for i,thisbar in enumerate(bar.patches):
    # Set a different hatch for each bar
    if i not in [5,6,7,8,9]:
        thisbar.set_facecolor(colors[c])
        c = c+1
ax.set_xticklabels(['1','2-6','7','>7'],rotation=0,fontsize=18)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.grid()
ax.set_xlim([-.75,3.75])
plt.legend([],frameon=False)
ax.set_ylim([0,650])
ax.set_ylabel('No. with mdSAMs',fontsize=18)
ax.set_xlabel('No. of TM segments',fontsize=18)
plt.savefig('TM_segments_v2.svg',bbox_inches='tight',transparent=True)


In [ ]:
number_of_TMs

In [ ]:
fig, ax = plt.subplots(figsize=(3.3,3.3)) 
sns.barplot(x='n',y='val',data=number_of_TMs,hue='ele',zorder=10)
# ax.grid()
# df_fisher = pd.DataFrame(l_fisher)
# df_fisher['fdr'] = fdr(df_fisher['pval'])
# df_fisher[df_fisher.fdr>.05]
fig, ax = plt.subplots(figsize=(2.5*1.618,2.5)) 
sns.barplot(x='n',y='val',data=df_fisher,hue='ele',zorder=10
           ,hue_order=['obs','exp'],lw=0,palette=['#2626A7','#939598'])
plt.ylim([300,550])
# ax.text(-.25,550,'***')
# ax.text(5.75,200,'***')
# ax.text(7.0,150,'*')
# plt.xlabel('No. TM segments',fontsize=10)
# plt.ylabel('No. with mdSAM',fontsize=10)
# ax.grid()
#plt.savefig('TM_segments.svg',bbox_inches='tight',transparent=True)

In [ ]:
uniport_annotation  = pd.read_csv('/Users/aelazar/Downloads/idmapping_2023_11_15.tsv',delimiter='\t')
OD = set(uniport_annotation[uniport_annotation['Gene Ontology IDs'].str.find('GO:0004984')!=-1].Entry)# PM GO 
t = hTMs['acc'].value_counts().reset_index()
mdSAMs_no_OD = mdSAMs[~mdSAMs.acc.isin(OD)]
t_no_OD = t[~t['index'].isin(OD)]
d = []
v = len(mdSAMs_no_OD.drop_duplicates(subset='acc')[mdSAMs_no_OD.no_TMs.between(1,1)])
d.append({'n':'1','val':v,'norm_v':v/len(mdSAMs_no_OD.drop_duplicates(subset='acc')),'ele':'mdSAM'})

v = len(mdSAMs_no_OD.drop_duplicates(subset='acc')[mdSAMs_no_OD.no_TMs.between(2,7)])
d.append({'n':'2-7','val':v,'norm_v':v/len(mdSAMs_no_OD.drop_duplicates(subset='acc')),'ele':'mdSAM'})

v = len(mdSAMs_no_OD.drop_duplicates(subset='acc')[mdSAMs_no_OD.no_TMs>7])
d.append({'n':'>7','val':v,'norm_v':v/len(mdSAMs_no_OD.drop_duplicates(subset='acc')),'ele':'mdSAM'})

v = len(t_no_OD[(~t_no_OD['index'].isin(mdSAMs_no_OD.acc))&(t_no_OD['acc'].between(1,1))])
d.append({'n':'1','val':v,'norm_v':v/len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)]),'ele':'non_mdSAM'})

v = len(t_no_OD[(~t_no_OD['index'].isin(mdSAMs_no_OD.acc))&(t_no_OD['acc'].between(2,7))])
d.append({'n':'2-7','val':v,'norm_v':v/len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)]),'ele':'non_mdSAM'})


v = len(t_no_OD[(~t_no_OD['index'].isin(mdSAMs_no_OD.acc))&(t_no_OD['acc'].between(7,100))])
d.append({'n':'>7','val':v,'norm_v':v/len(t_no_OD[~t_no_OD.index.isin(mdSAMs_no_OD.acc)]),'ele':'non_mdSAM'})
number_of_TMs = pd.DataFrame(d)
l_fisher = []
for i in set(number_of_TMs.n):
    a = number_of_TMs[(number_of_TMs.n==i)&(number_of_TMs.ele=='mdSAM')].val.values[0]
    b = number_of_TMs[(number_of_TMs.ele=='mdSAM')].val.sum()-a
    c = number_of_TMs[(number_of_TMs.n==i)&(number_of_TMs.ele=='non_mdSAM')].val.values[0]
    e = number_of_TMs[(number_of_TMs.ele=='non_mdSAM')].val.sum()-c
    table = [[a,b],[c,e]]
    res = fisher_exact(table)
    expected_freq(table)
    # print(i)
    # print(table[0][0],expected_freq(table)[0][0])
    # print(res)
    l_fisher.append({'n':i,'val':table[0][0],'ele':'obs','pval':res[-1]})
    l_fisher.append({'n':i,'val':expected_freq(table)[0][0],'ele':'exp','pval':res[-1]})
df_fisher = pd.DataFrame(l_fisher)
print(df_fisher)

In [ ]:
l = []
for i in ['1','2-7','>7']:
    obs = df_fisher[(df_fisher.n==i)&(df_fisher.ele=='obs')].val.values[0]
    exp = df_fisher[(df_fisher.n==i)&(df_fisher.ele=='exp')].val.values[0]
    enrich = np.log2(obs/exp)
    pval = df_fisher[(df_fisher.n==i)&(df_fisher.ele=='exp')].pval.values[0]
    l.append({'n':i,'enrich':enrich,'pval':pval})
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr
df_TM = pd.DataFrame(l)
df_TM['fdr'] = fdr(df_TM['pval'])
df_TM['fdr_log']= -1*np.log10(df_TM['fdr'])
fig, ax = plt.subplots(figsize=(3,4)) 
sns.scatterplot(x='n',y='enrich',data=df_TM,size='fdr_log')

In [ ]:
species = (
    " ")
weight_counts = {
    "1": np.array([543]),
    "2-7": np.array([286]),
    ">7": np.array([110]),
}
width = .1
colors=['#232323','#585858','#838383']
fig, ax = plt.subplots(figsize=(1,4)) 
bottom = np.zeros(1)
c = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c])
    bottom += weight_count
    c = c+1
plt.ylabel('No. proteins',fontsize=16)
plt.locator_params(nbins=5)
plt.yticks(fontsize=14, rotation=0)
#plt.savefig('donut_v3.svg',bbox_inches='tight',transparent=True)
sns.despine(ax=ax)
plt.savefig('Topology_bar_NO_OD.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_TM

In [ ]:
import matplotlib as mpl
norm = mpl.colors.Normalize(vmin=.5, vmax=35)

In [ ]:
l = []
for i in ['>7','2-7','1']:
    obs = df_fisher[(df_fisher.n==i)&(df_fisher.ele=='obs')].val.values[0]
    exp = df_fisher[(df_fisher.n==i)&(df_fisher.ele=='exp')].val.values[0]
    enrich = np.log2(obs/exp)
    pval = df_fisher[(df_fisher.n==i)&(df_fisher.ele=='exp')].pval.values[0]
    l.append({'n':i,'enrich':enrich,'pval':pval})
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr
df_TM = pd.DataFrame(l)
df_TM['fdr'] = fdr(df_TM['pval'])
df_TM['fdr_log']= -1*np.log10(df_TM['fdr'])
fig, ax = plt.subplots(figsize=(4,4)) 
sns.scatterplot(y='n',x='enrich',data=df_TM,hue='fdr_log',palette='Greys',s=100,edgecolor='k')
plt.axvline(0,ls='--',color='k')
ax.grid()
plt.ylim([-.5,2.5])
plt.xlabel('FC(log$_{2}$)',fontsize=16)
plt.ylabel('No. TM segments',fontsize=16)
sm = plt.cm.ScalarMappable(cmap='Greys',norm=norm)
#sm.set_array(df_TM['fdr_log'])
plt.legend([],frameon=False)
cbar =fig.colorbar(sm, ax=ax,location='right', shrink=0.4,aspect=10)
cbar.set_label('-log$_{10}FDR$', rotation=90)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
plt.savefig('Topology_FC_NO_OD.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.barplot(y='n',x='enrich',data=df_TM,order=['1','2-7','>7'])
plt.xlabel('FC(log$_{2}$)',fontsize=16)
plt.ylabel('No. TM segments',fontsize=16)
ax.grid()
plt.locator_params(nbins=4)
plt.axvline(0,ls='--',color='k')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
plt.savefig('Topology_FC_NO_OD_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
species = (
    " ")
weight_counts = {
    ">7": np.array([110]),
    "2-7": np.array([292]),
    "1": np.array([543]),
}
width = .1
colors=['#053A89','#397BDE','#232323']
fig, ax = plt.subplots(figsize=(1,4)) 
bottom = np.zeros(1)
c = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c])
    bottom += weight_count
    c = c+1
plt.ylabel('No. proteins',fontsize=16)
plt.locator_params(nbins=5)
plt.yticks(fontsize=14, rotation=0)
#plt.savefig('donut_v3.svg',bbox_inches='tight',transparent=True)
sns.despine(ax=ax)
plt.savefig('Topology_bar_All.svg',bbox_inches='tight',transparent=True)

In [ ]:
l = []
for i in range(7,10):
    l.append({'Length':'%d'%i,'Count':len(mdSAMs[mdSAMs.AA_length==i])})
l.append({'Length':'≥10','Count':len(mdSAMs[mdSAMs.AA_length>=10])})
print(l)
fig, ax = plt.subplots(figsize=(2.5,2.5))#
sns.barplot(x='Length',y='Count',data=pd.DataFrame(l),color='#939598',width=.95,lw=1)
ax.grid()
plt.xlabel('Length (Amino acids)',fontsize=12)
plt.savefig('mdSAMs_length_distribution.svg',bbox_inches='tight',transparent=True)

In [ ]:
mdSAMs

In [ ]:
import matplotlib 
import cmasher as cmr
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#5A75A3","#BB5450"])#"#"
colors = cmr.take_cmap_colors(ww, 5, return_fmt='hex')
md_length = pd.DataFrame(l)
md_length['pro']= 100*(md_length['Count']/np.sum( md_length['Count']))
md_length

In [ ]:
fig, ax = plt.subplots(figsize=(1.5,4)) 

sns.barplot(y=md_length['Length'],x=md_length['pro'],width=.85,color='#C6C6C8',alpha=1,zorder=2,lw=0)
plt.locator_params(nbins=4,axis='x')

ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.xlabel('')
plt.ylabel('')
ax.grid(zorder=0)
plt.savefig('mdSAMs_length_distribution.svg',bbox_inches='tight',transparent=True)


In [ ]:
weight_counts= dict(zip(md_length.Length,md_length.Count))
#weight_counts = dict(reversed(weight_counts.items()))
width = .1
# colors=['#053A89','#397BDE','#232323','#053A89','#397BDE','#232323','#053A89','#397BDE','#232323']
fig, ax = plt.subplots(figsize=(1,4)) 
bottom = np.zeros(1)
c = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c])
    bottom += weight_count
    c = c+1
plt.ylabel('No. proteins',fontsize=16)
plt.locator_params(nbins=4)
plt.yticks(fontsize=14, rotation=0)
plt.savefig('mdSAMs_length_distribution_stack.svg',bbox_inches='tight',transparent=True)

In [ ]:
dd = []
x = mdSAMs.acc.value_counts()
for i in range(1,3):
    v = len(x[x==i])
    dd.append({'n':'%d'%i,'val':v})
v = len(x[x>=3])
dd.append({'n':'$\\geq 3$','val':v})
# for i in range(1,8):
#     v = len(t[~t.index.isin(mdSAMs.acc)][t==i])
#     d.append({'n':'%d'%i,'val':v,'norm_v':v/len(t[~t.index.isin(mdSAMs.acc)]),'ele':'non_mdSAM'})
# v = len(t[~t.index.isin(mdSAMs.acc)][t>7])
# d.append({'n':'>7','val':v,'norm_v':v/len(t[~t.index.isin(mdSAMs.acc)]),'ele':'non_mdSAM'})

In [ ]:
#fig, ax = plt.subplots(figsize=(1.4,3)) 
fig, ax = plt.subplots(figsize=(3,4)) 
plt.locator_params(axis='y', nbins=5)
sns.barplot(x='n',y='val',data=pd.DataFrame(dd),zorder=10,palette=['#939598','#939598','#939598'],alpha=1,lw=0,width=.95)
ax.grid()
plt.ylabel('Count',fontsize=18)
#plt.xlabel('$No.\ mdSAMs\;in\;protein$',fontsize=10)
plt.xlabel('No mdSAMs in protein',fontsize=18)
ax.fill_between([1.5,3.5], 0,760 
               ,color='#B4B5B4', alpha=0.5)
#plt.locator_params(nbins=8)
plt.ylim([0,750])
plt.xlim([-.65,2.65])
plt.text(1.50,200,'<10%',fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=18)
plt.savefig('No_anaysis.svg',bbox_inches='tight',transparent=True)

In [ ]:
mdSAM_per_protein = pd.DataFrame(dd)

In [ ]:

weight_counts= dict(zip(mdSAM_per_protein.n,mdSAM_per_protein.val))
#weight_counts = dict(reversed(weight_counts.items()))
width = .1
colors=['#053A89','#397BDE','#232323','#053A89','#397BDE','#232323','#053A89','#397BDE','#232323']
fig, ax = plt.subplots(figsize=(1,4)) 
bottom = np.zeros(1)
c = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c])
    bottom += weight_count
    c = c+1
plt.ylabel('No. proteins',fontsize=16)
plt.locator_params(nbins=4)
plt.yticks(fontsize=14, rotation=0)
plt.savefig('No_anaysis_stacked.svg',bbox_inches='tight',transparent=True)

In [ ]:
mdSAM_per_protein
mdSAM_per_protein['pro']= 100*(mdSAM_per_protein['val']/np.sum( mdSAM_per_protein['val']))
mdSAM_per_protein

In [ ]:
np.sum( mdSAM_per_protein['val'])

In [ ]:
fig, ax = plt.subplots(figsize=(1.5,4)) 

sns.barplot(y=mdSAM_per_protein['n'],x=mdSAM_per_protein['pro'],width=.85,color='#C6C6C8',zorder=2)
plt.xlim([0,75])
plt.locator_params(nbins=4,axis='x')
ax.grid()
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.xlabel('')
plt.ylabel('')

plt.savefig('mdSAMs_number_distribution.svg',bbox_inches='tight',transparent=True)


In [ ]:
fig, ax = plt.subplots(figsize=(4,2)) 
sns.barplot(x='n',y='val',hue='ele',
            data=df_fisher,dodge=True,
            order=['1','2','3','4','5','6','7','>7'],zorder=10,palette=['#C03830','k'])
for i in enumerate(['1','2','3','4','5','6','7','>7']):
    p = df_fisher[(df_fisher.n==i[1])].drop_duplicates(subset='pval').pval.values[0]
    if p>.05:
        e='ns'
        plt.text(i[0],100,'$ns$'
                 ,rotation=90,fontsize=7,color='k',zorder=12)
    else:
        e = '%1.10e' % p
        lead = e[0]
        power = e.split('-')[-1].split('0')[-1]
        plt.text(i[0]+.02,10,'$p=%s⋅10^{‑%s}$'%(lead,power),rotation=90,fontsize=8,color='w',zorder=12)
ax.grid()
L=plt.legend(fontsize=8)
L.get_texts()[0].set_text('$Observed$')
L.get_texts()[1].set_text('$Expected$')
plt.ylabel('$Count$',fontsize=10)

plt.xlabel('$No.\;TM \;segments$',fontsize=10)
plt.savefig('TM_anaysis.svg',bbox_inches='tight',transparent=True)

In [ ]:

from mycolorpy import colorlist as mcp
color1=mcp.gen_color(cmap="Blues_r",n=8)
print(color1)

In [ ]:
plt.bar(df_fisher[df_fisher.ele=='obs'].sort_values(by='n').val,
        labels=df_fisher[df_fisher.ele=='obs'].sort_values(by='n').n,colors=color1)

In [ ]:
from matplotlib import colors
import matplotlib

In [ ]:
df_fisher.pivot(index='n',columns='ele',values='val')[::-1]['obs']/df_fisher.pivot(index='n',columns='ele',values='val')[::-1]['exp']

In [ ]:
df_fisher[]

In [ ]:
sns.set(font_scale=.75,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

divnorm=colors.LogNorm(vmin=10,vmax=600)
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#18827E","#B34F91"])#"#F3F3F3"
fig, ax = plt.subplots(figsize=(4,2)) 
sns.heatmap(df_fisher.pivot(index='n',columns='ele',values='val')[::-1],square=True,cmap=ww,cbar_kws={"shrink": .4,'location':'right'})
plt.ylabel('$No.\;TM \;segments$',fontsize=10)

plt.xlabel('')
ax.set_xticklabels(['$Exp$','$Obs$'],fontsize=8,rotation=90)
plt.savefig('TM_anaysis_pie.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4,2)) 
plt.pie(df_fisher[df_fisher.ele=='exp'].sort_values(by='n').val,
        labels=df_fisher[df_fisher.ele=='exp'].sort_values(by='n').n,colors=color1)

In [ ]:
df_fisher[df_fisher.ele=='obs'].sort_values(by='n').val.iloc[0]

In [ ]:
plt.bar('A', df_fisher[df_fisher.ele=='exp'].sort_values(by='n').val.iloc[0], color='#DC9538')
plt.bar('A', df_fisher[df_fisher.ele=='exp'].sort_values(by='n').val.iloc[1], bottom=df_fisher[df_fisher.ele=='exp'].sort_values(by='n').val.iloc[0], color='b')
plt.show()

In [ ]:
sns.barplot(x='val',data=df_fisher[df_fisher.ele=='obs'].sort_values(by='n'),hue='n')

In [ ]:
c = []
for acc in set(mdSAMs[mdSAMs.acc.isin(SSTM_topology.From)].acc):
    side1 = SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[1].split('"')[1]
    side2 = SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[-1].split('"')[1]
   
    c.append((side1,side2))

In [ ]:
ddd = []
x = mdSAMs.AA_length.value_counts()
for i in range(7,12):
    v = x.loc[i]
    ddd.append({'n':'%d'%i,'val':v})
v = x.loc[12:].sum()
ddd.append({'n':'$\\geq 12$','val':v})
# for i in range(1,8):
#     v = len(t[~t.index.isin(mdSAMs.acc)][t==i])
#     d.append({'n':'%d'%i,'val':v,'norm_v':v/len(t[~t.index.isin(mdSAMs.acc)]),'ele':'non_mdSAM'})
# v = len(t[~t.index.isin(mdSAMs.acc)][t>7])
# d.append({'n':'>7','val':v,'norm_v':v/len(t[~t.index.isin(mdSAMs.acc)]),'ele':'non_mdSAM'})

In [ ]:
ddd = []
x = mdSAMs.AA_length.value_counts()
for i in range(7,12):
    v = x.loc[i]
    for j in range(v):
        ddd.append({'val':i})
v = x.loc[12:].sum()
for j in range(v):
    ddd.append({'val':12})

In [ ]:
fig, ax = plt.subplots(figsize=(5,2)) 

sns.boxplot(x=mdSAMs.AA_length,color='#C03830',saturation=.9)
ax.grid()
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=18)
plt.xlabel('Legnth (Amino acids)',fontsize=20)
plt.savefig('No_AA_anaysis_box.svg',bbox_inches='tight',transparent=True)

In [ ]:
v

In [ ]:
#fig, ax = plt.subplots(figsize=(1.4,3)) 
fig, ax = plt.subplots(figsize=(4,4)) 
plt.locator_params(nbins=4)
sns.barplot(x='n',y='val',data=pd.DataFrame(ddd),zorder=10,palette=['#949597','#949597','#949597'],lw=0)
ax.grid()
plt.ylabel('$Count$',fontsize=10)
plt.ylabel('Count',fontsize=10)

#plt.xlabel('$mdSAM\;legnth\,(Amino\;acids)$',fontsize=10)
plt.xlabel('mdSAM legnth (Amino acids)',fontsize=10)


ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=18)
plt.savefig('No_AA_anaysis.svg',bbox_inches='tight',transparent=True)

In [ ]:
pd.DataFrame(ddd)

In [ ]:
1- (1425-481-473-251)/pd.DataFrame(ddd).val.sum()

In [ ]:
c

In [ ]:
cc = []
for acc in SSTM_topology.From:
    side1 = SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[1].split('"')[1]
    side2 = SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[-1].split('"')[1]
    cc.append((side1,side2))

In [ ]:
mdSAMs.sample(1)

In [ ]:
def random_loc_list(ind):

    acc = mdSAMs.loc[ind,'acc']
    #print(acc)
    side = SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[1].split('"')[1]
    if SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('TOPO_DOM')[1].split('..')[0][1:].isnumeric()==True:
        s = int(SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('TOPO_DOM')[1].split('..')[0][1:])
        e = int(SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('TOPO_DOM')[1].split('..')[1].split(';')[0])
        #mdSAMs[mdSAMs.acc==acc].loc[4727,'start_new']                                     

        #print('N term is %s'%side,s,e)                                                                                                        
        #print('TM starts @ %s ends @ %s'%(hTMs[hTMs.acc==acc].start.values[0],hTMs[hTMs.acc==acc].end.values[0]))  


        r,temp = random_idr_in_mdSAM_seg(ind)
        mdSAM_s = r+1
        #print('mdSAM starts @ %d'%mdSAM_s)
        if mdSAM_s in range(s,e+1):
            dist= hTMs[hTMs.acc==acc].end.values[0]- mdSAM_s
            loc = SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[1].split('"')[1]
        else:
            dist = (mdSAM_s -hTMs[hTMs.acc==acc].end.values[0])
            loc = (SSTM_topology[SSTM_topology.From==acc]['Topological domain'].values[0].split('note=')[-1].split('"')[1])
        #print(loc,dist)
        return loc,dist
    else:
        return np.NaN,np.NaN

In [ ]:
rand_l = []

for i in range(10):
    for i in mdSAMs[mdSAMs.acc.isin(SSTM_topology.From)].index:
        acc = mdSAMs.loc[i,'acc']
        r_l,r_d = random_loc_list(i)
    #print(loc_,dist_)
        rand_l.append({'location':r_l,'dist':r_d,'acc':acc})

In [ ]:
ran_idrs = pd.DataFrame(rand_l)

In [ ]:
sns.ecdfplot(x='dist',data=ran_idrs[(ran_idrs.location.isin(['Cytoplasmic']))&(ran_idrs.dist>0)],log_scale=(True))
sns.ecdfplot(x='dist',data=mdSAMs[(mdSAMs.location.isin(['Cytoplasmic']))&(mdSAMs.dist>0)]
                  ,log_scale=(True),c='orange')#,palette=['#E67D79','#8CC6EC','#C0C0BE']

In [ ]:
ran_idrs[(ran_idrs.location.isin(['Cytoplasmic']))&(ran_idrs.dist>0)].dist.median()
#mdSAMs[(mdSAMs.location.i|sin(['Cytoplasmic']))&(mdSAMs.dist>0)].dist.median()

In [ ]:
stats.ks_2samp(ran_idrs[(ran_idrs.location.isin(['Cytoplasmic']))&(ran_idrs.dist>0)].dist,
               mdSAMs[(mdSAMs.location.isin(['Cytoplasmic']))&(mdSAMs.dist>0)].dist)

In [ ]:
ran_idrs

In [ ]:

z = []
for c in ['Extracellular']:#,'Cytoplasmic''Extracellular','Lumenal'
    for i in mdSAMs[mdSAMs.location.isin(['%s'%c])].index:#,
        loca = mdSAMs.loc[i,'location']
        dist = mdSAMs.loc[i,'dist']
        acc = mdSAMs.loc[i,'acc']
        rand_median = ran_idrs[(ran_idrs.acc==acc)&(ran_idrs.dist>0)&(ran_idrs.location==loca)].dist.mean()
        #print(rand_median)
        if np.isnan(rand_median)==False:
            if dist>0:
                z.append({'val':rand_median-dist,'c':i,'ele':'real'})
                #z.append({'val':dist,'c':i,'ele':'real'})
                #z.append({'val':rand_median,'c':i,'ele':'random'})


In [ ]:
zz= pd.DataFrame(z)
print(zz)
#sns.scatterplot(x='val',y='rand_median',data=zz[zz.val>0],edgecolor=None,alpha=.1)
# plt.ylim([0,500])
# plt.xlim([0,500])
# # plt.axhline(.0)
# plt.plot([500,0],[500,0])

In [ ]:
mdSAMs.head(1)
def ranges(nums):
    nums = sorted(set(nums))
    gaps = [[s, e] for s, e in zip(nums, nums[1:]) if s+1 < e]
    edges = iter(nums[:1] + sum(gaps, []) + nums[-1:])
    return list(zip(edges, edges))

In [ ]:
def random_idr_in_mdSAM_seg(ind):
    acc = mdSAMs.loc[ind,'acc']
    s = mdSAMs.loc[ind,'start_new']
    rs = ranges([i[0] for i in enumerate(AF2[AF2.acc==acc].pLDDT.values[0]) if i[1]<70])
    for i in rs:
        if s in range(i[0],i[1]+1):
            mdsam_r = pd.Series(list(range(i[0],i[1]))).sample(1).values[0]
            len_ = len(pd.Series(list(range(i[0],i[1]))))
            #(mdsam_r)
    return mdsam_r,len_

In [ ]:
random_idr_in_mdSAM_seg(6)

In [ ]:
mdSAMs#.loc[i,'acc']

In [ ]:
mdSAMs[mdSAMs.acc=='Q9NQW8']

In [ ]:
hTMs[hTMs.acc=='Q9NQW8'].iloc[0]
hTMs[hTMs.acc=='Q9NQW8'].iloc[-1]


In [ ]:
hTMs[hTMs.acc=='Q6DN72'].sort_values(by='start')#.iloc[0]['start']

In [ ]:
def mdSAM_loc(ind):
    acc = mdSAMs.loc[ind,'acc']
    idr_res = set([i[0]+1 for i in enumerate(AF2[AF2.acc==acc].pLDDT.values[0]) if i[1]<70 and i[0]>30])
    
    n_term = set(range(1,hTMs[hTMs.acc==acc].sort_values(by='start').iloc[0]['start']))
    idr_n = set.intersection(n_term,idr_res)
    
    c_term  = set(range(hTMs[hTMs.acc==acc].sort_values(by='start').iloc[-1]['end'],len(AF2[AF2.acc==acc].AAseq.values[0])))
    idr_c = set.intersection(c_term,idr_res)
    r_mdSAM = set(range(int(mdSAMs.loc[ind,'start_new']),int(mdSAMs.loc[ind,'end_new'])))
    idr_loop = set.difference(idr_res,set.union(idr_c,idr_n))
    print(idr_n)
    print(idr_c)
    print(idr_loop)
    print(len(set.intersection(r_mdSAM,idr_c)))
    if len(set.intersection(r_mdSAM,n_term))>0:
        return('n_term')
    if len(set.intersection(r_mdSAM,c_term))>0:
        return('c_term')
    if (len(set.intersection(r_mdSAM,c_term))==0)and(len(set.intersection(r_mdSAM,n_term))==0):
        return('loop')   

In [ ]:
mdSAMs.start_new.min()

In [ ]:
def mdSAM_loc_v2(acc):
    #acc = mdSAMs.loc[ind,'acc']
    TMs = []
    for i in hTMs[hTMs.acc==acc].index:
        for res in range(hTMs.loc[i,'start'],hTMs.loc[i,'end']+1):
            TMs.append(res)
    TMs_set = set(TMs)
    idr_res_1 = set.difference(set([i[0]+1 for i in enumerate(AF2[AF2.acc==acc].pLDDT.values[0]) if i[1]<70]),TMs_set)
    idr_res_2 = set.difference(set([i[0]+1 for i in enumerate(AF2[AF2.acc==acc].SS.values[0]) if i[1]!='H']),TMs_set)
    idr_res_3 = set.difference(set([i[0]+1 for i in enumerate(AF2[AF2.acc==acc].accessibility.values[0]) if i[1]=='*']),TMs_set)
    idr_res = set.intersection(idr_res_1,idr_res_2,idr_res_3)
    
    n_term = set.difference(set(range(1,hTMs[hTMs.acc==acc].sort_values(by='start').iloc[0]['start'])),set(range(0,0)))
    idr_n = set.intersection(n_term,idr_res)
    
    c_term  = set(range(hTMs[hTMs.acc==acc].sort_values(by='start').iloc[-1]['end'],len(AF2[AF2.acc==acc].AAseq.values[0])))
    idr_c = set.intersection(c_term,idr_res)
    r_mdSAM_list = []
    for ind in mdSAMs[mdSAMs.acc==acc].index: 
        for res in set(range(int(mdSAMs.loc[ind,'start_new']),int(mdSAMs.loc[ind,'end_new']))):
            r_mdSAM_list.append(res)
    r_mdSAM = set(r_mdSAM_list)
    
    idr_loop = set.difference(idr_res,set.union(idr_c,idr_n))
    #print(idr_loop)
    # (len(idr_n))
    # print(len(idr_c))
    # print(len(idr_loop))
    # print(len(set.intersection(r_mdSAM,idr_c)))
    # print(len(set.intersection(r_mdSAM,idr_n)))
    # print(len(set.intersection(r_mdSAM,idr_loop)))
    return {'acc':acc,'n_all':len(idr_n),'c_all':len(idr_c),'loop_all':len(idr_loop)-1
            ,'n_mdSAM':len(set.intersection(r_mdSAM,idr_n)),'c_mdSAM':len(set.intersection(r_mdSAM,idr_c)),'loop_mdSAM':len(set.intersection(r_mdSAM,idr_loop))}
    # if len(set.intersection(r_mdSAM,n_term))>0:
    #     return('n_term')
    # if len(set.intersection(r_mdSAM,c_term))>0:
    #     return('c_term')
    # if (len(set.intersection(r_mdSAM,c_term))==0)and(len(set.intersection(r_mdSAM,n_term))==0):
    #     return('loop')   

In [ ]:
l  =[]
for acc in set(mdSAMs.acc):
    l.append(mdSAM_loc_v2(acc))
loc_agg = pd.DataFrame(l)

In [ ]:
l = []
for i in loc_agg.index:
    val = loc_agg.loc[i,'n_all'] - loc_agg.loc[i,'n_mdSAM'] 
    l.append({'val':val,'topology':'n','mdSAM':'-'})
    l.append({'val':loc_agg.loc[i,'n_mdSAM'],'topology':'n','mdSAM':'+'})
    val = loc_agg.loc[i,'c_all'] - loc_agg.loc[i,'c_mdSAM'] 
    l.append({'val':val,'topology':'c','mdSAM':'-'})
    l.append({'val':loc_agg.loc[i,'c_mdSAM'],'topology':'c','mdSAM':'+'})
    val = loc_agg.loc[i,'loop_all'] - loc_agg.loc[i,'loop_mdSAM'] 
    l.append({'val':val,'topology':'loop','mdSAM':'-'})
    l.append({'val':loc_agg.loc[i,'loop_mdSAM'],'topology':'loop','mdSAM':'+'})


In [ ]:
chi_ = pd.DataFrame(l)

In [ ]:
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr

In [ ]:
def termi_enrich(topo):
    total_runners =  chi_.val.sum() #  No. of Non-hTMs Nodes in the Graph 
    number_team_A =  chi_[(chi_.topology==topo)].val.sum()# number Nodes that are in Endo-data and in Gragh  
    number_winners = chi_[(chi_.mdSAM=='+')].val.sum()
    number_win_from_A = chi_[(chi_.topology==topo)&(chi_.mdSAM=='+')].val.sum()
    expected = (number_team_A*number_winners)/total_runners
    #print(total_runners,number_team_A,number_winners,number_win_from_A,expected)
    p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return {'topo':topo,'enrich':np.log2(number_win_from_A/expected),'obs':number_win_from_A,'exp':expected,'pval':p}
print(termi_enrich('n'))

In [ ]:
l = []
for topo in ['n','c','loop']:
    l.append(termi_enrich(topo))

In [ ]:
topo_enrich = pd.DataFrame(l)
topo_enrich['FDR'] = [ -1*np.log10(i) for i in fdr(topo_enrich.pval)]
topo_enrich

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.barplot(y='topo',x='enrich',data=topo_enrich,order=['c','n','loop'],zorder=2)
plt.xlabel('FC(log$_{2}$)',fontsize=16)
plt.ylabel('mdSAM res',fontsize=16)
ax.grid(zorder=0)
plt.xlim([-.6,.6])
plt.locator_params(nbins=4)
plt.axvline(0,ls='--',color='k')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)

plt.savefig('Topology__locFC.svg',bbox_inches='tight',transparent=True)

In [ ]:
d5.location.value_counts()

In [ ]:
species = (
    " ")
weight_counts = {
    "c_term": np.array([124]),
    "n_term": np.array([493]),
    "loop": np.array([808]),
}
width = .1
colors=['#053A89','#397BDE','#232323']
fig, ax = plt.subplots(figsize=(1,4)) 
bottom = np.zeros(1)
c = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c])
    bottom += weight_count
    c = c+1
plt.ylabel('No. proteins',fontsize=16)
plt.locator_params(nbins=5)
plt.yticks(fontsize=14, rotation=0)
#plt.savefig('donut_v3.svg',bbox_inches='tight',transparent=True)
#sns.despine(ax=ax)
plt.savefig('Topology_loc_bar_All.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
sns.barplot(x='ele',y='val',palette=['#C6C6C8','#04347B','#BB5450'],
            data=pd.DataFrame([{'val':808,'ele':'c_term'},{'val':493,'ele':'n_term'},{'val':124,'ele':'loop'}]))
plt.ylabel('No. proteins',fontsize=16)
plt.locator_params(nbins=4)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
ax.grid()
plt.savefig('Topology_loc_bar_All_horz.svg',bbox_inches='tight',transparent=True)

In [ ]:
a = chi_[(chi_.topology=='n')&(chi_.mdSAM=='-')].val.sum()
b = chi_[(chi_.topology=='c')&(chi_.mdSAM=='-')].val.sum()
c = chi_[(chi_.topology=='loop')&(chi_.mdSAM=='-')].val.sum()

In [ ]:
aa = chi_[(chi_.topology=='n')&(chi_.mdSAM=='+')].val.sum()
bb = chi_[(chi_.topology=='c')&(chi_.mdSAM=='+')].val.sum()
cc = chi_[(chi_.topology=='loop')&(chi_.mdSAM=='+')].val.sum()

In [ ]:
np.log2([[aa, bb,cc], [a, b,c]]/chi2_contingency(table,lambda_="log-likelihood")[-1])

In [ ]:
[aa,bb,cc]
import numpy as np
from scipy.stats import chi2_contingency
table = np.array([[aa, bb,cc], [a, b,c]])
res = chi2_contingency(table,lambda_="log-likelihood")
res

In [ ]:
d4 = []
for i in mdSAMs.index:
    d4.append({'acc':mdSAMs.loc[i,'acc'],'AAseq':mdSAMs.loc[i,'AAseq'],'location':(mdSAM_loc(i))})

In [ ]:
d5 = pd.DataFrame(d4)#.location.value_counts()

In [ ]:
d5.location.value_counts()

In [ ]:
124/(808+493+124)

In [ ]:
d5[d5.location=='loop'].drop_duplicates('acc').acc.to_clipboard(index=0)

In [ ]:
mdSAMs[mdSAMs.AAseq=='LNVNIYRH']

In [ ]:
import csv

GPCRdb = pd.read_csv('/Users/aelazar/notebooks_for_paper/GPCR_ThT/Book1.txt',delimiter='\t',quoting=csv.QUOTE_NONE, error_bad_lines=False) # https://wiki.stjude.org/display/C3D/GPCR+common+data

In [ ]:
x =d5.location.value_counts()

In [ ]:
d5[d5.acc.isin(GPCRdb['#UniProt accession'])]#.location.value_counts()

In [ ]:
d5[(d5.acc.isin(GPCRdb['#UniProt accession']))].location.value_counts()

In [ ]:
species = (
    " ")
weight_counts = {
    ">7": np.array([110]),
    "2-7": np.array([292]),
    "1": np.array([543]),
}
width = .1
colors=['#053A89','#397BDE','#232323']
fig, ax = plt.subplots(figsize=(1,4)) 
bottom = np.zeros(1)
c = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c])
    bottom += weight_count
    c = c+1
plt.ylabel('No. proteins',fontsize=16)
plt.locator_params(nbins=5)
plt.yticks(fontsize=14, rotation=0)
#plt.savefig('donut_v3.svg',bbox_inches='tight',transparent=True)
sns.despine(ax=ax)
plt.savefig('Topology_bar_All.svg',bbox_inches='tight',transparent=True)

In [ ]:
#fig, ax = plt.subplots(figsize=(1.4,3)) 
fig, ax = plt.subplots(figsize=(3,4)) 
plt.locator_params(nbins=4)
sns.barplot(x=['c_term','n_term','loop'],y=[808,493,124],zorder=0,palette=['#053A89','#397BDE','#232323'],width=.95)
ax.grid(zorder=0)
plt.ylabel('Count',fontsize=16)
plt.xlabel('mdSAM location',fontsize=16)
ax.set_xticklabels(["C' tail ","N' tail","Loop"])
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)

plt.savefig('No_loc_topoloy_anaysis.svg',bbox_inches='tight',transparent=True)

In [ ]:
#fig, ax = plt.subplots(figsize=(1.4,3)) 
fig, ax = plt.subplots(figsize=(4.,4)) 

sns.barplot(x=['Termini', 'Loops'],y=[493+808, 124],zorder=10,palette=['#053A89','#C43E96'])
ax.grid()
plt.ylabel('Count',fontsize=16)
plt.xlabel('Location',fontsize=16)
ax.set_xticklabels(["Termini","Loop",],rotation=90)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
plt.savefig('No_loc_topoloy_anaysis.svg',bbox_inches='tight',transparent=True)

In [ ]:
MUNV940103 ={'A':1.08,'L':0.789,'R':0.976,'K':1.026,'N':1.197,'M':.812,'D':1.266,'F':0.685,
             'C':.733,'P':1.412,'Q':1.05,'S':0.987,'E':1.085,'T':0.784,'G':1.104,'W':0.755,'H':0.906,
             'Y':0.665,'I':0.583,'V':0.546}#from AAindex MUNV940103
from scipy import stats

In [ ]:
waltzdb = pd.read_csv('/Users/aelazar/Downloads/waltzdb_export.csv')#http://waltzdb.switchlab.org/ 
waltzdb = waltzdb.drop_duplicates(subset=['Sequence'])
waltzdb =waltzdb.reset_index()

In [ ]:
df = pd.DataFrame()
dff = pd.DataFrame()
for k in waltzdb.index:
    if k%500==0:
        print(k)
    seq = waltzdb.loc[k,'Sequence']
    if waltzdb.loc[k,'Classification']=='amyloid':
        classif = 1
    else:
        classif = 0 
    dff = dff.append({'classification':classif,'beta':np.sum([MUNV940103[i] for i in waltzdb.loc[k,'Sequence']])},ignore_index=True)
df = pd.concat([dff, pd.get_dummies(waltzdb['Sequence'].apply(lambda x:pd.Series(list(x))))], axis=1)#One-hot encodong 

In [ ]:
df = df.iloc[np.random.permutation(len(df))]#shuffling the data
df = df.reset_index()
df = df.drop(['index'],axis=1)

In [ ]:
df_t = df 
df_t.loc[df_t[df_t.classification==1].index,'type'] = 'Amyloid'
df_t.loc[df_t[df_t.classification==0].index,'type'] = 'Non-Amyloid'
print(stats.ttest_ind(df_t[(df_t['type']=='Amyloid')].beta,df_t[(df_t['type']=='Non-Amyloid')].beta))
fig = plt.figure(figsize=(5, 7))
ax = sns.boxplot(x='type',y='beta',data=df_t,notch=True,palette=['#B4B5B4','#9D282B'],width=.5,linewidth=2,showfliers=False)
ax = sns.stripplot(x="type", y="beta", data=df_t, color="0",alpha=0.1)
ax.set(xlabel=None)
#ax.text(0.5, 0.05,'Text Here', fontsize=9)
#plt.text(0.1, 0.9, 'AUC=.88', size=15)#, color='purple')
sns.despine(offset=10, trim=True)
#plt.axhline(y=.6, linewidth=1,ls='--', color='k')
plt.ylabel(r'$\sum\Delta G_{\beta} $')#'Free energy in beta-strand conformation'
adjust_box_widths(fig, 1.4)
x1, x2 = 0, 1   # columns 'Sat' and 'Sun' (first column: 0, see plt.xticks())
y, h, col = df_t['beta'].max() + .1, .1, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "$p=10^{-112}$", ha='center', va='bottom', color=col)

plt.tight_layout()
#plt.savefig('Delta_G_beta_paper.png',dpi=1200)

In [ ]:
fig, ax = plt.subplots(figsize=(2.5, 2.5)) 
sns.set(font_scale=1,style='white')
ax.grid()
sns.kdeplot(hue='type',x='beta',data=df_t,hue_order=['Non-Amyloid','Amyloid'],palette=['#C03830','#B4B5B4'],common_norm=False,fill=True,alpha=1)
plt.xlabel(r'$\sum\Delta G_{\beta}(MUNV940103)$',fontsize=10)#'Free energy in beta-strand conformation'
plt.ylabel(r'$Density$',fontsize=10)
plt.legend(['$Non‑Amyloid$','Amyloid'],fontsize=8)
plt.xlim([3.5,8])
e = '%1.10e' % stats.ttest_ind(df_t[(df_t['type']=='Amyloid')].beta,df_t[(df_t['type']=='Non-Amyloid')].beta)[-1]
lead = e[0]
power = e.split('-')[-1].split('0')[-1]
plt.text(5,.05,'$p=%s⋅10^{‑%s}$'%(lead,power),rotation=0,fontsize=10,color='w')
plt.savefig('Delta_G_beta_kde.svg',bbox_inches='tight',transparent=False)

In [ ]:
df_t